# Description

Reads drug-disease prediction results from the `011-prediction-*` notebooks and computes
final performance measures (AUROC, average precision) against the PharmacotherapyDB gold standard.

There are 10 prediction files in total:
- 5 module-based (all LVs, top-5, top-10, top-25, top-50)
- 5 gene-based (all genes, top-50, top-100, top-250, top-500)

Aggregation per method:
1. Average prediction scores across all n_top thresholds for each drug-disease pair.
2. These averaged scores are the final predictions.

(No tissue aggregation needed since we use a single multi-tissue SMulTiXcan result.)

# Module loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from collections import defaultdict
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

from pyprojroot import here

# Settings

In [3]:
# 1 data source (SMulTiXcan), 5 n_top thresholds per method, 2 methods
N_THRESHOLDS = 5
N_METHODS = 2
N_PREDICTION_FILES = N_THRESHOLDS * N_METHODS  # 10

In [4]:
DATA_DIR = here('data/archs4/drug_diseases_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

OUTPUT_PREDICTIONS_DIR = here('output/drug_disease_analyses') / 'predictions' / 'dotprod_neg'
display(OUTPUT_PREDICTIONS_DIR)
assert OUTPUT_PREDICTIONS_DIR.exists()

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/archs4/drug_diseases_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/dotprod_neg')

# Load PharmacotherapyDB gold standard

In [5]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


In [6]:
gold_standard['true_class'].value_counts()

true_class
1    755
0    243
Name: count, dtype: int64

In [7]:
gold_standard['true_class'].value_counts(normalize=True)

true_class
1    0.756513
0    0.243487
Name: proportion, dtype: float64

# Load drug-disease predictions

In [8]:
current_prediction_files = list(OUTPUT_PREDICTIONS_DIR.glob('*.h5'))
display(len(current_prediction_files))
assert len(current_prediction_files) == N_PREDICTION_FILES, (
    f'Expected {N_PREDICTION_FILES} files, found {len(current_prediction_files)}'
)

10

In [9]:
display(sorted([f.name for f in current_prediction_files]))

['smultixcan-mashr-zscores-data-all_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-data-top_100_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-data-top_250_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-data-top_500_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-data-top_50_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-projection-all_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-projection-top_10_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-projection-top_25_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-projection-top_50_genes-prediction_scores.h5',
 'smultixcan-mashr-zscores-projection-top_5_genes-prediction_scores.h5']

In [10]:
predictions = []

for f in tqdm(current_prediction_files, ncols=100):
    # Load predictions and merge with gold standard (keep only pairs present there)
    prediction_data = pd.read_hdf(f, key='prediction')
    prediction_data = pd.merge(
        prediction_data, gold_standard, on=['trait', 'drug'], how='inner'
    )

    # Convert scores to ranks
    prediction_data['score'] = prediction_data['score'].rank()
    prediction_data['trait'] = prediction_data['trait'].astype('category')
    prediction_data['drug'] = prediction_data['drug'].astype('category')

    # Add metadata columns
    metadata = pd.read_hdf(f, key='metadata')
    prediction_data = prediction_data.assign(method=metadata.method.values[0])
    prediction_data['method'] = prediction_data['method'].astype('category')
    prediction_data = prediction_data.assign(n_top_genes=metadata.n_top_genes.values[0])
    prediction_data = prediction_data.assign(data=metadata.data.values[0])
    prediction_data['data'] = prediction_data['data'].astype('category')

    predictions.append(prediction_data)

predictions = pd.concat(predictions, ignore_index=True)

100%|███████████████████████████████████████████████████████████████| 10/10 [00:01<00:00,  7.93it/s]


In [11]:
display(predictions.shape)
display(predictions.head())

(6850, 7)

,trait,drug,score,true_class,method,n_top_genes,data
0,DOID:0050741,DB00215,263.0,1,Gene-based,-1.0,smultixcan-mashr-zscores-data
1,DOID:0050741,DB00704,222.0,1,Gene-based,-1.0,smultixcan-mashr-zscores-data
2,DOID:0050741,DB00822,586.0,1,Gene-based,-1.0,smultixcan-mashr-zscores-data
3,DOID:10283,DB00014,114.0,1,Gene-based,-1.0,smultixcan-mashr-zscores-data
4,DOID:10283,DB00175,18.0,0,Gene-based,-1.0,smultixcan-mashr-zscores-data


In [12]:
assert not predictions.isna().any().any()

In [13]:
_tmp = predictions['method'].value_counts()
display(_tmp)
n_unique_pairs = predictions[['drug', 'trait']].drop_duplicates().shape[0]
assert _tmp.loc['Gene-based'] == N_THRESHOLDS * n_unique_pairs
# rough check: same count per method

method
Gene-based      3425
Module-based    3425
Name: count, dtype: int64

## Save raw predictions

In [14]:
output_file = OUTPUT_PREDICTIONS_DIR.parent / 'predictions_results.pkl'
predictions.to_pickle(output_file)
print(f'Saved to: {output_file}')

Saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/predictions_results.pkl


# Aggregate predictions

Average prediction scores across all n_top thresholds for each (trait, drug, method) combination.
These are the final drug-disease predictions used for performance evaluation.

In [15]:
def _reduce_mean(x):
    return pd.Series({
        'score': x['score'].mean(),
        'true_class': x['true_class'].unique()[0]
    })


predictions_avg = (
    predictions
    .groupby(['trait', 'drug', 'method'])
    .apply(_reduce_mean)
    .dropna()
    .sort_index()
    .reset_index()
)

/tmp/ipykernel_511870/1415174730.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['trait', 'drug', 'method'])
/tmp/ipykernel_511870/1415174730.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_reduce_mean)


In [16]:
display(predictions_avg.shape)
display(predictions_avg.head())

(1370, 5)

,trait,drug,method,score,true_class
0,DOID:0050741,DB00215,Gene-based,294.2,1.0
1,DOID:0050741,DB00215,Module-based,313.8,1.0
2,DOID:0050741,DB00704,Gene-based,277.4,1.0
3,DOID:0050741,DB00704,Module-based,279.2,1.0
4,DOID:0050741,DB00822,Gene-based,536.6,1.0


In [17]:
assert predictions_avg.dropna().shape == predictions_avg.shape

## Save aggregated predictions

In [18]:
output_file = OUTPUT_PREDICTIONS_DIR.parent / 'predictions_results_aggregated.pkl'
predictions_avg.to_pickle(output_file)
print(f'Saved to: {output_file}')

Saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/drug_disease_analyses/predictions/predictions_results_aggregated.pkl


# ROC performance

In [19]:
# AUROC per method and n_top_genes threshold
predictions.groupby(['method', 'n_top_genes']).apply(
    lambda x: roc_auc_score(x['true_class'], x['score'])
).rename('AUROC').to_frame()

/tmp/ipykernel_511870/2264405014.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  predictions.groupby(['method', 'n_top_genes']).apply(


AUROC
method       n_top_genes          
Gene-based   -1.0         0.564819
              50.0        0.550071
              100.0       0.544238
              250.0       0.546781
              500.0       0.563474
Module-based -1.0         0.538820
              5.0         0.562104
              10.0        0.554094
              25.0        0.537157
              50.0        0.551673

In [20]:
# Final AUROC using averaged predictions
auroc_final = predictions_avg.groupby('method').apply(
    lambda x: roc_auc_score(x['true_class'], x['score'])
).rename('AUROC')
display(auroc_final)

/tmp/ipykernel_511870/2344677460.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  auroc_final = predictions_avg.groupby('method').apply(


method
Gene-based      0.562251
Module-based    0.551447
Name: AUROC, dtype: float64

# Average Precision (PR-AUC) performance

In [21]:
# Average precision per method and n_top_genes threshold
predictions.groupby(['method', 'n_top_genes']).apply(
    lambda x: average_precision_score(x['true_class'], x['score'])
).rename('AvgPrecision').to_frame()

/tmp/ipykernel_511870/1794767312.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  predictions.groupby(['method', 'n_top_genes']).apply(


AvgPrecision
method       n_top_genes              
Gene-based   -1.0             0.816331
              50.0            0.816429
              100.0           0.818378
              250.0           0.818544
              500.0           0.820323
Module-based -1.0             0.820449
              5.0             0.824628
              10.0            0.821420
              25.0            0.818583
              50.0            0.824033

In [22]:
# Final average precision using averaged predictions
ap_final = predictions_avg.groupby('method').apply(
    lambda x: average_precision_score(x['true_class'], x['score'])
).rename('AvgPrecision')
display(ap_final)

/tmp/ipykernel_511870/3724490884.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ap_final = predictions_avg.groupby('method').apply(


method
Gene-based      0.820159
Module-based    0.824363
Name: AvgPrecision, dtype: float64

# Summary

In [23]:
summary = pd.concat([auroc_final, ap_final], axis=1)
display(summary)

,AUROC,AvgPrecision
method,,
Gene-based,0.562251,0.820159
Module-based,0.551447,0.824363
